In [28]:
import pandas as pd

til_new = pd.read_excel("data/tilflytning.xlsx",header=None)
fra_new = pd.read_excel("data/fraflytning.xlsx",header=None)

til_old = pd.read_excel("data/tilflytningold.xlsx",header=None)
fra_old = pd.read_excel("data/fraflytningold.xlsx",header=None)

def clean_dst(df, kommune_col, year_start_col, prefix=None, value_name="value"):
    df = df.copy()

    # Years are in row 2
    years = df.iloc[2, year_start_col:].tolist()

    # Data starts in row 3
    df = df.iloc[3:].copy()

    # Rename kommune column
    df = df.rename(columns={kommune_col: "kommune"})

    # Rename year columns
    year_cols_old = list(df.columns[year_start_col:])
    rename_years = dict(zip(year_cols_old, years))
    df = df.rename(columns=rename_years)

    # Keep only kommune rows
    df = df[df["kommune"].notna()].copy()

    # Clean kommune names
    df["kommune"] = df["kommune"].astype(str).str.strip()

    if prefix is not None:
        df = df[df["kommune"].str.startswith(prefix + " ")].copy()
        df["kommune"] = df["kommune"].str.replace(prefix + " ", "", regex=False)

    # Year columns
    year_cols = [c for c in df.columns if pd.notna(c) and str(c).replace(".0", "").isdigit()]

    # Wide to long
    df = df.melt(
        id_vars=["kommune"],
        value_vars=year_cols,
        var_name="år",
        value_name=value_name
    )

    df["år"] = pd.to_numeric(df["år"], errors="coerce").astype(int)
    df[value_name] = pd.to_numeric(df[value_name], errors="coerce").fillna(0)

    # Sum over age groups
    df = df.groupby(["kommune", "år"], as_index=False)[value_name].sum()

    return df

In [29]:
til_new = clean_dst(
    til_new,
    kommune_col=2,
    year_start_col=3,
    prefix="Til",
    value_name="tilflytning"
)

fra_new = clean_dst(
    fra_new,
    kommune_col=2,
    year_start_col=3,
    prefix="Fra",
    value_name="fraflytning"
)

til_old = clean_dst(
    til_old,
    kommune_col=1,
    year_start_col=2,
    prefix=None,
    value_name="tilflytning"
)

fra_old = clean_dst(
    fra_old,
    kommune_col=2,
    year_start_col=3,
    prefix=None,
    value_name="fraflytning"
)


In [30]:
fra_new = fra_new[(fra_new["år"] >= 1992) & (fra_new["år"] <= 2025)].copy()
til_new = til_new[(til_new["år"] >= 1992) & (til_new["år"] <= 2025)].copy()
til_old = til_old[(til_old["år"] >= 1992) & (til_old["år"] <= 2025)].copy()
fra_old = fra_old[(fra_old["år"] >= 1992) & (fra_old["år"] <= 2025)].copy()

fra_new_pivot = fra_new.pivot(
    index="år",
    columns="kommune",
    values="fraflytning"
)

til_new_pivot = til_new.pivot(
    index="år",
    columns="kommune",
    values="tilflytning"
)

fra_old_pivot = fra_old.pivot(
    index="år",
    columns="kommune",
    values="fraflytning"
)

til_old_pivot = til_old.pivot(
    index="år",
    columns="kommune",
    values="tilflytning"
)   

In [31]:
import pandas as pd

# ---------- Helper ----------
def clean_kommune_series(s):
    return (
        s.astype(str)
         .str.replace(r"\s*\(.*?\)", "", regex=True)
         .str.strip()
         .str.replace("Nykøbing-Falster", "Nykøbing Falster", regex=False)
    )

# ---------- Load mapping ----------
mapping = pd.read_excel(
    "data/Korrespondancetabel-mellem-kommuner-foer-og-efter-kommunalreformen-i-2007.xlsx"
)

mapping = mapping[["AMT_KOM_TXT", "NUTS_TXT"]].copy()
mapping.columns = ["old_kommune", "new_kommune"]

# Clean mapping names
mapping["old_kommune"] = clean_kommune_series(mapping["old_kommune"])
mapping["new_kommune"] = clean_kommune_series(mapping["new_kommune"])

# Fix special cases
mapping["new_kommune"] = mapping["new_kommune"].replace({
    "Bornholm excl. Christiansø": "Bornholm",
    "Christiansø Uden for Kommuner": None,
    "Ærø 2005/2006-": "Ærø"
})

# Drop Christiansø
mapping = mapping[mapping["new_kommune"].notna()]

# Remove duplicates (important!)
mapping = mapping.drop_duplicates(subset=["old_kommune"], keep="first")
# ---------- Manual special-case mappings ----------
manual_mapping = pd.DataFrame({
    "old_kommune": [
        "Aakirkeby",
        "Allinge-Gudhjem",
        "Hasle",
        "Nexø",
        "Rønne",
        "Bornholm",
        "Ærøskøbing",
        "Marstal",
        "Ærø"
    ],
    "new_kommune": [
        "Bornholm",
        "Bornholm",
        "Bornholm",
        "Bornholm",
        "Bornholm",
        "Bornholm",
        "Ærø",
        "Ærø",
        "Ærø"
    ]
})

mapping = pd.concat([mapping, manual_mapping], ignore_index=True)
mapping = mapping.drop_duplicates(subset=["old_kommune"], keep="last")

# ---------- Clean old migration data ----------
til_old["kommune"] = clean_kommune_series(til_old["kommune"])
fra_old["kommune"] = clean_kommune_series(fra_old["kommune"])

# ---------- Merge mapping ----------
til_old_mapped = til_old.merge(
    mapping,
    left_on="kommune",
    right_on="old_kommune",
    how="left"
)

fra_old_mapped = fra_old.merge(
    mapping,
    left_on="kommune",
    right_on="old_kommune",
    how="left"
)

# ---------- Debug missing mappings ----------
missing = til_old_mapped[til_old_mapped["new_kommune"].isna()]["kommune"].unique()
print("Missing mappings:", missing)

# ---------- Replace kommune with new ----------
til_old_mapped["kommune"] = til_old_mapped["new_kommune"]
fra_old_mapped["kommune"] = fra_old_mapped["new_kommune"]

# ---------- Aggregate ----------
til_old_mapped = til_old_mapped.groupby(
    ["kommune", "år"], as_index=False
)["tilflytning"].sum()

fra_old_mapped = fra_old_mapped.groupby(
    ["kommune", "år"], as_index=False
)["fraflytning"].sum()



Missing mappings: ['Christiansø']


In [32]:
# ---------- Drop rows that failed mapping ----------
til_old_mapped = til_old_mapped[til_old_mapped["kommune"].notna()].copy()
fra_old_mapped = fra_old_mapped[fra_old_mapped["kommune"].notna()].copy()

# ---------- Combine old + new ----------
til_all = pd.concat(
    [til_old_mapped[["kommune", "år", "tilflytning"]], 
     til_new[["kommune", "år", "tilflytning"]]],
    ignore_index=True
)

fra_all = pd.concat(
    [fra_old_mapped[["kommune", "år", "fraflytning"]], 
     fra_new[["kommune", "år", "fraflytning"]]],
    ignore_index=True
)

# ---------- Aggregate just in case there are duplicates ----------
til_all = til_all.groupby(["kommune", "år"], as_index=False)["tilflytning"].sum()
fra_all = fra_all.groupby(["kommune", "år"], as_index=False)["fraflytning"].sum()

# ---------- Merge til/fra ----------
migration = til_all.merge(
    fra_all,
    on=["kommune", "år"],
    how="outer"
)

migration["tilflytning"] = migration["tilflytning"].fillna(0)
migration["fraflytning"] = migration["fraflytning"].fillna(0)

# ---------- Net migration ----------
migration["net_migration"] = (
    migration["tilflytning"] - migration["fraflytning"]
)

# ---------- Sort ----------
migration = migration.sort_values(["kommune", "år"]).reset_index(drop=True)

migration.head()


migration = migration[migration["kommune"] != "Christiansø"].copy()

migration["kommune"] = migration["kommune"].replace({
    "Lyngby-Tårbæk": "Lyngby-Taarbæk"
})




In [33]:
population = pd.read_csv("data/population_clean.csv")

# ---------- Merge ----------
migration = migration.merge(
    population,
    on=["kommune", "år"],
    how="left"
)

# ---------- Check ----------
print("Missing population:", migration["population"].isna().sum())

# ---------- Migration rate ----------
migration["migration_rate"] = (
    migration["net_migration"] / migration["population"]
)

migration["migration_per_1000"] = (
    migration["migration_rate"] * 1000
)

# ---------- Sort ----------
migration = migration.sort_values(["kommune", "år"]).reset_index(drop=True)

migration.to_csv("data/migration_population_clean.csv", index=False)

Missing population: 0
